# Colab runner — unimodal-transformer

Thin launcher: this notebook clones the repo onto the Colab VM and drives the modular `src/` code on a GPU.

**Before running:** `Runtime → Change runtime type → T4 GPU`.

Every cell below runs on the Google VM, not your Mac. The VM is ephemeral — it is wiped when the runtime resets — so you re-run the clone/install cell at the start of each session, and anything you want to keep must be pushed to GitHub or saved to Drive (last section).

## 1. Check the GPU

In [ ]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU — set Runtime > Change runtime type > T4 GPU")

## 2. Clone the repo onto the VM and install deps

Idempotent: clones on first run, `git pull` on re-runs within the same session. Colab preinstalls torch/numpy/scipy/matplotlib, so the install is fast.

In [ ]:
import os

REPO_URL = "https://github.com/slocke3/unimodal-transformer.git"
REPO_DIR = "/content/unimodal-transformer"

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR

%cd $REPO_DIR
!git pull --ff-only
!pip install -q -r requirements.txt
print("\nrepo contents:")
!ls

## 2b. (Optional) Restore a previous run from Drive

On a fresh runtime `outputs/` is empty (the VM was wiped). If you already trained and saved to Drive, this copies that run's `checkpoints/` + `figures/` back onto the VM — so you can skip retraining and go straight to eval / Figure 1. Set `RUN_STAMP` to the timestamped folder you saved. (Alternatively, just run the training cell below to regenerate everything.)

In [ ]:
from google.colab import drive
import os, shutil

RUN_STAMP  = "base_20260609-173050"   # timestamped Drive folder from a previous save
DRIVE_BASE = "/content/drive/MyDrive/unimodal-transformer-outputs"

drive.mount("/content/drive")
src_dir = os.path.join(DRIVE_BASE, RUN_STAMP)
assert os.path.isdir(src_dir), f"{src_dir} not found — check RUN_STAMP / DRIVE_BASE"

for sub in ("checkpoints", "figures"):
    s = os.path.join(src_dir, sub)
    if os.path.isdir(s):
        os.makedirs(f"outputs/{sub}", exist_ok=True)
        shutil.copytree(s, f"outputs/{sub}", dirs_exist_ok=True)
        print(f"restored {sub}: {os.listdir(f'outputs/{sub}')}")
    else:
        print(f"(skip) {s} not found")

## 3. Smoke test

Tiny end-to-end run (50 trajectories, small model, 1 epoch) to confirm data → model → train loop executes on the GPU before launching a real run.

In [ ]:
from src.dataset import make_splits
from src.model import DiscreteTrajectoryTransformer
from src.trainer import Trainer, TrainerConfig

train_loader, val_loader, _ = make_splits(
    n_trajectories=50, context_len=20, n_bins=32, traj_len=80,
    batch_size=128, num_workers=2, seed=0,
)
model = DiscreteTrajectoryTransformer(
    n_bins=32, context_len=20, d_model=64, n_heads=2, n_layers=2,
)
cfg = TrainerConfig(max_epochs=1, patience=5, save_dir="outputs/checkpoints")
trainer = Trainer(model, train_loader, val_loader, config=cfg, run_name="smoke")
print("device:", trainer.device)
history = trainer.train()
print("smoke OK | best val loss:", history["best_val_loss"])

## 4. Full training run

Drives the CLI with the base config. Saves checkpoint, history, and resolved config under `outputs/checkpoints/`.

In [ ]:
!python train.py --config configs/base.yaml --run_name base

## 5. Evaluate

Per-r cross-entropy vs Lyapunov, cross-family generalization, and rollout plots → `outputs/figures/`.

In [ ]:
!python evaluate.py \
    --checkpoint outputs/checkpoints/base_best.pt \
    --config outputs/checkpoints/base_config.yaml

## 5b. View figures inline (no Drive trip needed)

`evaluate.py` runs as a subprocess, so it saves PNGs to `outputs/figures/` but can't display them. This cell renders every figure right here in the notebook — so you check results in the tab you're already in, without opening Drive.

In [ ]:
import glob
from IPython.display import Image, display

figs = sorted(glob.glob("outputs/figures/*.png"))
if not figs:
    print("No figures found — run the Evaluate cell first.")
for path in figs:
    print(path)
    display(Image(filename=path))

## 5c. Figure 1 — transformer CE vs empirical $k$-gram oracle vs $\lambda$

Compares the transformer's per-parameter cross-entropy against a **per-system empirical $k$-gram oracle**, fitted on separate trajectories from each evaluated system, and the qualified **CE = $\lambda^+$** reference. The oracle is target-informed rather than a zero-shot competitor. Both are scored on identical test contexts in two panels: in-distribution (quadratic) and out-of-family (zero-shot transformer evaluation on tent/sine/cubic).

Takes a few minutes — most of the time is the $\lambda$ computation for the out-of-family panel (progress prints as it goes). Needs `base_best.pt` on this VM (run training, or restore from Drive in section 2b).

In [ ]:
import os, torch
from src.model import DiscreteTrajectoryTransformer
from src.baseline_compare import run_figure1

CKPT = "outputs/checkpoints/base_best.pt"
if not os.path.exists(CKPT):
    raise FileNotFoundError(
        CKPT + " not found on this VM. Run the training cell, or restore from "
        "Drive in section 2b.")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = DiscreteTrajectoryTransformer(n_bins=64, context_len=50,
                                      d_model=128, n_heads=4, n_layers=4)
sd = torch.load(CKPT, map_location=device, weights_only=True)
model.load_state_dict(sd["model_state_dict"])
model.to(device).eval()

fig, in_dist, fam_data = run_figure1(
    model, device, n_bins=64, context_len=50,
    kgram_orders=(1, 2, 5), n_eval=20, n_params=100, lam_steps=20000,
    save_path="outputs/figures/figure1_kgram_vs_lambda.png",
)
fig  # display inline

## 5d. N×L sweep (bin count × context length)

Trains all 12 (N, L) models, saving each checkpoint and its CE(r) curve to Drive
**incrementally** (a disconnect only loses the in-flight run; re-running resumes).
Tests the Markov / temporal-locality hypothesis: is CE flat in L (context doesn't
help) at coarse N, and does it fan out at fine N? Mount Drive first.

In [ ]:
import os, json, time, shutil, yaml, torch
from google.colab import drive
from src.dataset import make_splits
from src.model import DiscreteTrajectoryTransformer
from src.trainer import Trainer, TrainerConfig
from src.sweep import make_sweep_configs, ce_vs_r, save_ce_result, DEFAULT_NS, DEFAULT_LS

drive.mount("/content/drive")
SWEEP_DIR = "/content/drive/MyDrive/unimodal-transformer-outputs/sweep_NxL"
os.makedirs(SWEEP_DIR, exist_ok=True)
device = "cuda"

base = yaml.safe_load(open("configs/base.yaml"))
runs = make_sweep_configs(base, ns=DEFAULT_NS, ls=DEFAULT_LS)

res_path = os.path.join(SWEEP_DIR, "sweep_ce_results.json")
done = json.load(open(res_path)) if os.path.exists(res_path) else {}
print(f"{len(done)}/{len(runs)} runs already done")

for run_name, cfg in runs:
    if run_name in done:
        print("skip", run_name); continue
    t0 = time.time()
    d, m, tr = cfg["data"], cfg["model"], cfg["training"]
    train_loader, val_loader, _ = make_splits(
        n_trajectories=d["n_trajectories"], r_range=tuple(d["r_range"]),
        context_len=d["context_len"], n_bins=d["n_bins"], traj_len=d["traj_len"],
        burn_in=d["burn_in"], train_frac=d["train_frac"], val_frac=d["val_frac"],
        batch_size=d["batch_size"], num_workers=d["num_workers"], seed=d["seed"])
    model = DiscreteTrajectoryTransformer(
        n_bins=d["n_bins"], context_len=d["context_len"], d_model=m["d_model"],
        n_heads=m["n_heads"], n_layers=m["n_layers"], d_ff=m.get("d_ff"), dropout=m["dropout"])
    tcfg = TrainerConfig(lr=tr["lr"], weight_decay=tr["weight_decay"],
        max_epochs=tr["max_epochs"], patience=tr["patience"], grad_clip=tr["grad_clip"],
        log_every=tr["log_every"], eval_every=tr["eval_every"], save_dir="outputs/checkpoints")
    trainer = Trainer(model, train_loader, val_loader, config=tcfg, run_name=run_name)
    print(f"=== {run_name} (N={d['n_bins']}, L={d['context_len']}) ===")
    trainer.train()
    trainer.load_best()
    shutil.copy(f"outputs/checkpoints/{run_name}_best.pt",
                os.path.join(SWEEP_DIR, f"{run_name}_best.pt"))
    rg, ce = ce_vs_r(model, device, n_bins=d["n_bins"], context_len=d["context_len"])
    save_ce_result(SWEEP_DIR, run_name, d["n_bins"], d["context_len"], rg, ce)
    print(f"  {run_name} done in {time.time()-t0:.0f}s")

print("SWEEP COMPLETE")

## 5e. Plot the sweep

One panel per N, one curve per L (raw CE, each panel its own y-scale). Look for
the L-curves collapsing onto each other (context doesn't matter → ~Markov),
and whether that collapse weakens as N grows.

In [ ]:
import json
from src.sweep import plot_sweep
from IPython.display import Image, display

results = json.load(open(os.path.join(SWEEP_DIR, "sweep_ce_results.json")))
fig = plot_sweep(results, save_path="outputs/figures/sweep_NxL.png")
display(Image("outputs/figures/sweep_NxL.png"))

## 5f. System-ID probe (implied return map vs context length)

Loads the N=64 sweep checkpoints at L=2,4,8,32 and reads each model's implied
one-step map E[x_{n+1}|x_n] on contexts from a fixed r, comparing to the true
parabola f_r. Tests whether the L=2 sweep deficit is a system-ID failure.
Needs the sweep checkpoints in Drive (run 5d first, or a prior sweep).

In [ ]:
import os, torch
from google.colab import drive
from src.model import DiscreteTrajectoryTransformer
from src.system_id import run_system_id, plot_return_map_grid, plot_deviation_and_bias

drive.mount("/content/drive")
SWEEP_DIR = "/content/drive/MyDrive/unimodal-transformer-outputs/sweep_NxL"
device, N = "cuda", 64
Ls = [2, 4, 8, 32]

models = {}
for L in Ls:
    m = DiscreteTrajectoryTransformer(n_bins=N, context_len=L,
                                      d_model=128, n_heads=4, n_layers=4)
    ckpt = f"{SWEEP_DIR}/sweep_N{N}_L{L}_best.pt"
    sd = torch.load(ckpt, map_location=device, weights_only=True)
    m.load_state_dict(sd["model_state_dict"]); m.to(device).eval()
    models[L] = m
    print("loaded", ckpt)

res = run_system_id(models, device, n_bins=N, r_values=[3.6, 3.7, 3.8, 3.9, 4.0])
plot_return_map_grid(res, save_path="outputs/figures/system_id_grid.png")
plot_deviation_and_bias(res, save_path="outputs/figures/system_id_dev_bias.png")

from IPython.display import Image, display
for f in ("system_id_grid.png", "system_id_dev_bias.png"):
    display(Image(f"outputs/figures/{f}"))

## 5g. OOD blow-up: occupancy diagnostic + conjugacy CE fix

Shows the tent OOD blow-up is a coordinate artifact. (1) occupancy diagnostic
(torch-free): 78% of raw-tent transitions are unseen in logistic training,
0% after the conjugacy h. (2) model-side CE: the base model's CE on raw tent
vs h(tent) vs logistic r=4 -- expect raw tent >> h(tent) ~ logistic r=4.
Needs base_best.pt (run cell 2b to restore, or cell 4 to retrain).

In [ ]:
import torch
from src.occupancy import run_occupancy_diagnostic, plot_occupancy, conjugacy_ce_test
from src.model import DiscreteTrajectoryTransformer
from IPython.display import Image, display

# (1) occupancy diagnostic (torch-free)
res = run_occupancy_diagnostic(n_bins=64)
plot_occupancy(res, save_path='outputs/figures/occupancy_ood.png')
display(Image('outputs/figures/occupancy_ood.png'))
print(f"unseen transitions -- raw tent: {res['unseen_tent']:.1%}  |  h(tent): {res['unseen_tent_h']:.1%}")

# (2) model-side CE confirmation on the base checkpoint
device = 'cuda'
model = DiscreteTrajectoryTransformer(n_bins=64, context_len=50,
                                      d_model=128, n_heads=4, n_layers=4)
sd = torch.load('outputs/checkpoints/base_best.pt', map_location=device, weights_only=True)
model.load_state_dict(sd['model_state_dict']); model.to(device).eval()

ce = conjugacy_ce_test(model, device, n_bins=64, eval_context=30)
print(f"\nBase-model CE (nats):")
print(f"  raw tent    : {ce['raw_tent']:.2f}")
print(f"  h(tent)     : {ce['h_tent']:.2f}")
print(f"  logistic r=4: {ce['logistic_r4']:.2f}   <- reference")

## 6. Persist outputs to Drive (the VM is ephemeral!)

`outputs/` is gitignored and lives only on the throwaway VM, so it vanishes when the runtime resets. The cell below mounts your Google Drive and copies `checkpoints/` + `figures/` into a **timestamped** folder (so repeated runs don't overwrite each other).

**Getting them onto your Mac:** install **Google Drive for desktop** once, and this folder syncs into a local folder you can open straight from VS Code / Finder — no manual download. Anything in Drive is also reachable from [drive.google.com](https://drive.google.com). Set `RUN_NAME` to match the `--run_name` you trained with.

In [ ]:
import os, shutil, datetime
from google.colab import drive

RUN_NAME   = "base"  # match the --run_name used in training
DRIVE_BASE = "/content/drive/MyDrive/unimodal-transformer-outputs"

drive.mount("/content/drive")

stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
dest  = os.path.join(DRIVE_BASE, f"{RUN_NAME}_{stamp}")
os.makedirs(dest, exist_ok=True)

for sub in ("checkpoints", "figures"):
    src = os.path.join("outputs", sub)
    if os.path.isdir(src) and os.listdir(src):
        shutil.copytree(src, os.path.join(dest, sub), dirs_exist_ok=True)
        print(f"copied {src}/  ->  {dest}/{sub}")
    else:
        print(f"(skip) {src}/ is empty or missing")

print("\nSaved to Drive:", dest)

Quick alternative for grabbing a single file straight to your browser's downloads (no Drive needed):

In [ ]:
# from google.colab import files
# files.download("outputs/figures/figure1_kgram_vs_lambda.png")